# TP 5 — Apprentissage supervisé : k-NN, classifieur de Bayes, validation croisée, métriques

**Cours :** *Supervised Learning* (slides 118–135) — risque et risque empirique, fonctions de perte, fonction de régression et **k-NN régression**, **classifieur de Bayes** et **k-NN classification**, avantages et limites du k-NN, **validation croisée**, **évaluation** d'un classifieur (matrice de confusion, précision/rappel, F-score, ROC).

**À la fin de ce notebook, vous saurez :**
1. coder les prédicteurs **k-NN** (régression, classification, probabilités) et relier $k$ au compromis biais-variance ;
2. calculer le **classifieur de Bayes** quand la loi est connue, et situer tout classifieur par rapport à l'**erreur de Bayes** ;
3. coder les principales **validations croisées** (holdout, $r$-fold, leave-one-out, Monte-Carlo) et comparer la variance de leurs estimations ;
4. coder la **matrice de confusion**, la **précision**, le **rappel**, le **F-score** et la courbe **ROC / AUC** ;
5. appliquer le tout à une question de **prévision de volatilité** — en évitant la fuite d'information propre aux séries temporelles.

> 📝 **Version énoncé.** Complétez les blocs `# TODO`, puis exécutez la cellule *Test* qui suit chaque exercice. Le corrigé est dans `../corriges/`.

In [ ]:
import sys, pathlib, warnings
_here = pathlib.Path.cwd()
ROOT = _here.parent if _here.name in ("exercices", "corriges") else _here
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tp_utils import *          # style graphique, check(), check_close(), timer()...
import tp_data                  # données réelles (cache / téléchargement) ou simulées

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
rng = np.random.default_rng(42)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score

## Partie 1 — Le cadre

On observe $(x_i, y_i)$ iid selon une loi jointe de $(X, Y)$ et on cherche $f$ minimisant le **risque** $\mathbb E[L(Y, f(X))]$ ; en pratique on minimise le **risque empirique** $\frac1n\sum L(y_i, f(x_i))$. Le choix de la perte définit la méthode :

| perte $L(y, f(x))$ ($y\in\{-1,1\}$) | nom | classifieur associé |
|---|---|---|
| $\mathbb 1_{y\ne f(x)}$ | 0-1 | Bayes, k-NN |
| $(1-yf(x))^2$ | quadratique | (régression linéaire, TP 6–7) |
| $\max\{0, 1-yf(x)\}$ | hinge | SVM (TP 7) |
| $\exp(-yf(x))$ | exponentielle | boosting (TP 8) |
| $\log(1+\exp(-yf(x)))$ | logistique | régression logistique (TP 7) |

In [ ]:
m = np.linspace(-2, 2.5, 400)
losses = {"0-1": (m <= 0).astype(float), "quadratique": (1 - m) ** 2, "hinge": np.maximum(0, 1 - m),
          "exponentielle": np.exp(-m), "logistique": np.log2(1 + np.exp(-m))}
fig, ax = plt.subplots(figsize=(7, 4))
for i, (k, v) in enumerate(losses.items()):
    ax.plot(m, v, label=k, color=PALETTE[i])
ax.set_ylim(0, 4); ax.set_xlabel("marge y·f(x)"); ax.set_ylabel("perte")
ax.set_title("Les pertes convexes majorent la perte 0-1 (logistique en base 2)"); ax.legend()
plt.show()

## Partie 2 — Régression : fonction de régression et k-NN

Avec la perte quadratique, le prédicteur optimal est la **fonction de régression** $f^*(x) = \mathbb E[Y\mid X = x]$. Le k-NN l'estime par une moyenne **locale** :
$\hat f(x) = \frac1k\sum_{x_i\in N_k(x)} y_i$. Il est **consistant** si $k\to\infty$ et $k/n\to0$.

### Exercice 2.1 — Voisins et prédicteurs k-NN
`knn_indices(Xtr, Xte, k)` renvoie la matrice $(m, k)$ des indices des $k$ plus proches voisins de chaque point test (distance euclidienne, calcul matriciel par blocs pour limiter la mémoire). On en déduit la régression, la classification (vote majoritaire) et la probabilité de la classe 1.

In [ ]:
def knn_indices(Xtr, Xte, k, block=2000):
    Xtr = np.asarray(Xtr, float).reshape(len(Xtr), -1)
    Xte = np.asarray(Xte, float).reshape(len(Xte), -1)
    out = np.empty((len(Xte), k), dtype=int)
    sq_tr = (Xtr ** 2).sum(1)
    for s in range(0, len(Xte), block):
        B = Xte[s:s + block]
        # TODO : distances² par blocs, argpartition, puis tri des k colonnes
        raise NotImplementedError("À compléter")
    return out


def knn_regress(Xtr, ytr, Xte, k):
    # TODO : moyenne des y des voisins renvoyés par knn_indices
    raise NotImplementedError("À compléter")


def knn_classify(Xtr, ytr, Xte, k):
    """Vote majoritaire (étiquettes entières 0..K-1 ; égalité -> plus petite étiquette)."""
    # TODO : compter les votes de chaque classe parmi les voisins, puis argmax
    raise NotImplementedError("À compléter")


def knn_proba(Xtr, ytr, Xte, k):
    """Fraction de voisins de classe 1 (estimation de P(Y=1 | X=x))."""
    # TODO : fraction des voisins de classe 1
    raise NotImplementedError("À compléter")

In [ ]:
# Tests vs scikit-learn
Xtr = rng.normal(size=(500, 3)); ytr_r = Xtr[:, 0] + rng.normal(0, 0.1, 500); ytr_c = (Xtr[:, 1] > 0).astype(int)
Xte = rng.normal(size=(200, 3))
check_close(knn_regress(Xtr, ytr_r, Xte, 7), KNeighborsRegressor(7).fit(Xtr, ytr_r).predict(Xte), "knn_regress = sklearn")
check(np.mean(knn_classify(Xtr, ytr_c, Xte, 7) == KNeighborsClassifier(7).fit(Xtr, ytr_c).predict(Xte)) == 1,
      "knn_classify = sklearn (k impair, 2 classes)")
check_close(knn_proba(Xtr, ytr_c, Xte, 7), KNeighborsClassifier(7).fit(Xtr, ytr_c).predict_proba(Xte)[:, 1],
            "knn_proba = sklearn")

### Expérience 2.2 — $k$ règle le compromis biais-variance

In [ ]:
f_true = lambda x: np.sin(2 * x) + 0.3 * x
x = rng.uniform(-3, 3, 200); y = f_true(x) + rng.normal(0, 0.4, 200)
xs = np.linspace(-3, 3, 400)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, k in zip(axes, [1, 15, 100]):
    ax.scatter(x, y, s=8, color=GRAY)
    ax.plot(xs, f_true(xs), color=INK, lw=1, ls="--", label="E[Y|X=x]")
    ax.plot(xs, knn_regress(x, y, xs, k), color=BLUE, label=f"{k}-NN")
    ax.set_title({1: "k = 1 : variance forte", 15: "k = 15 : compromis", 100: "k = 100 : biais fort"}[k]); ax.legend()
plt.tight_layout(); plt.show()

## Partie 3 — Classification : classifieur de Bayes et k-NN

Avec la perte 0-1, le prédicteur optimal est le **classifieur de Bayes** $f^*(x) = \arg\max_y \mathbb P(Y = y\mid X = x)$ ; son risque (l'**erreur de Bayes**) est le minimum atteignable par **n'importe quel** classifieur. Le k-NN estime $\mathbb P(Y=y\mid X=x)$ par la fraction des voisins de classe $y$.

On utilise le jeu jouet du cours (« mixture of 2 Gaussians », $n = 100 + 100$) : chaque classe est un mélange de 10 gaussiennes. Comme la loi est **connue**, on peut calculer le classifieur de Bayes exactement (`tp_data.ESLMixture`).

In [ ]:
mix = tp_data.ESLMixture()
Xtr, ytr = mix.sample(100, seed=1)
Xte, yte = mix.sample(5000, seed=2)                 # grand échantillon test ≈ vrai risque
bayes_err = mix.bayes_error()
print(f"erreur de Bayes ≈ {bayes_err:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
plot_decision_boundary(axes[0], mix.bayes_predict, Xtr, ytr, f"Bayes (erreur {bayes_err:.1%})")
for ax, k in zip(axes[1:], [1, 15]):
    err = np.mean(knn_classify(Xtr, ytr, Xte, k) != yte)
    plot_decision_boundary(ax, lambda Z, k=k: knn_classify(Xtr, ytr, Z, k), Xtr, ytr, f"{k}-NN (erreur test {err:.1%})")
axes[0].legend(loc="upper left")
plt.tight_layout(); plt.show()

### Expérience 3.1 — Erreur d'apprentissage, erreur de test, erreur de Bayes

In [ ]:
ks = [1, 3, 5, 7, 11, 15, 21, 31, 45, 69, 99, 151]
tr_err = [np.mean(knn_classify(Xtr, ytr, Xtr, k) != ytr) for k in ks]
te_err = [np.mean(knn_classify(Xtr, ytr, Xte, k) != yte) for k in ks]
fig, ax = plt.subplots()
ax.semilogx(ks, tr_err, marker="o", label="erreur d'apprentissage")
ax.semilogx(ks, te_err, marker="o", label="erreur de test")
ax.axhline(bayes_err, color=GRAY, ls="--", lw=1); ax.text(1, bayes_err + 0.005, "erreur de Bayes", color=INK2, fontsize=9)
ax.set_xlabel("k (← complexe | simple →)"); ax.set_ylabel("taux d'erreur"); ax.legend()
ax.set_title("k-NN : surapprentissage (k petit) et sous-apprentissage (k grand)")
plt.show()

**À retenir :** l'erreur d'apprentissage est un mauvais juge (nulle pour $k=1$). Il faut estimer l'erreur de généralisation sur des données **non vues** : c'est le rôle de la validation croisée.

## Partie 4 — Validation croisée

On partitionne les données en apprentissage $T$ et validation $V$ disjoints, et on moyenne l'erreur sur $V$ sur plusieurs partitions.
- **exhaustives** : *leave-p-out* ($\binom np$ partitions), **leave-one-out** (LOO, $n$ partitions) ;
- **non exhaustives** : **$r$-fold** (partition aléatoire en $r$ blocs, chacun sert une fois de validation), **holdout** (une seule partition), **Monte-Carlo** (partitions aléatoires répétées).

### Exercice 4.1 — Générateurs de partitions et estimateur CV
Chaque générateur renvoie une liste de couples `(idx_train, idx_val)`.

In [ ]:
def kfold_splits(n, r, rng):
    # TODO : permutation, np.array_split en r blocs ; chaque bloc sert une fois de validation
    raise NotImplementedError("À compléter")


def loo_splits(n):
    # TODO : n partitions (tous sauf i, [i])
    raise NotImplementedError("À compléter")


def holdout_split(n, frac_val, rng):
    # TODO : permutation ; les round(frac_val·n) premiers indices en validation ; renvoyer une liste d'un couple
    raise NotImplementedError("À compléter")


def monte_carlo_splits(n, frac_val, B, rng):
    # TODO : B appels à holdout_split
    raise NotImplementedError("À compléter")


def cv_error(fit_predict, X, y, splits):
    """Erreur de validation moyenne (pondérée par la taille des blocs)."""
    # TODO : erreur de validation de chaque partition, moyenne pondérée par la taille des blocs
    raise NotImplementedError("À compléter")

In [ ]:
# Tests
spl = kfold_splits(103, 10, rng)
allv = np.concatenate([v for _, v in spl])
check(len(spl) == 10 and np.array_equal(np.sort(allv), np.arange(103)), "r-fold : chaque point validé exactement une fois")
check(all(len(np.intersect1d(t, v)) == 0 for t, v in spl), "r-fold : apprentissage et validation disjoints")
check(len(loo_splits(20)) == 20, "LOO : n partitions")

### Exercice 4.2 — Choisir $k$ par validation croisée 10-fold

In [ ]:
fp = lambda k: (lambda A, b, C: knn_classify(A, b, C, k))
ks_cv = [1, 3, 5, 7, 9, 11, 15, 21, 31, 45, 69]
# TODO : mêmes 10 blocs pour toutes les valeurs de k (comparaison appariée)
raise NotImplementedError("À compléter")
err_star = np.mean(knn_classify(Xtr, ytr, Xte, k_star) != yte)
print(f"k choisi par CV : {k_star} | erreur de test : {err_star:.3f} | Bayes : {bayes_err:.3f}")
check(err_star < bayes_err + 0.08, "le k choisi par CV généralise correctement")

### Expérience 4.3 — Biais et variance des estimateurs de validation croisée

Pour 100 échantillons d'apprentissage indépendants ($n = 200$), on estime l'erreur du 15-NN par holdout (30 %), 5-fold, 10-fold, LOO et Monte-Carlo (20 tirages de 30 %), et on compare au **vrai** risque (mesuré sur le grand échantillon test). Pour le k-NN, LOO est peu coûteux : il suffit d'exclure chaque point de ses propres voisins.

In [ ]:
def loo_knn_error(X, y, k):
    idx = knn_indices(X, X, k + 1)[:, 1:]          # le 1er voisin de x_i est x_i lui-même
    votes = y[idx]
    pred = (votes.mean(1) > 0.5).astype(int)
    return np.mean(pred != y)

rows = []
for rep in range(100):
    r_ = np.random.default_rng(500 + rep)
    X, y = mix.sample(100, seed=500 + rep)
    true = np.mean(knn_classify(X, y, Xte, 15) != yte)
    rows.append({"holdout 30 %": cv_error(fp(15), X, y, holdout_split(200, 0.3, r_)) - true,
                 "5-fold": cv_error(fp(15), X, y, kfold_splits(200, 5, r_)) - true,
                 "10-fold": cv_error(fp(15), X, y, kfold_splits(200, 10, r_)) - true,
                 "LOO": loo_knn_error(X, y, 15) - true,
                 "Monte-Carlo (20 × 30 %)": cv_error(fp(15), X, y, monte_carlo_splits(200, 0.3, 20, r_)) - true})
gap = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(9, 3.8))
for i, c in enumerate(gap.columns):
    ax.scatter(gap[c], i + rng.uniform(-0.15, 0.15, len(gap)), s=10, color=PALETTE[i], alpha=0.6)
    ax.plot([gap[c].mean()] * 2, [i - 0.3, i + 0.3], color=INK, lw=2)
ax.axvline(0, color=GRAY, lw=1, ls="--")
ax.set_yticks(range(gap.shape[1]), gap.columns); ax.set_xlabel("estimation CV − vrai risque (trait noir : moyenne)")
ax.set_title("Holdout : très variable ; r-fold / LOO / Monte-Carlo : bien plus précis")
plt.show()
gap.agg(["mean", "std"]).T.rename(columns={"mean": "biais", "std": "écart-type"})

## Partie 5 — Évaluer un classifieur

Sur un ensemble de test, pour la classe « positive » :
- **précision** $= \frac{TP}{TP+FP}$ (fiabilité des alertes), **rappel** (TPR) $= \frac{TP}{TP+FN}$ (capacité à détecter), **fall-out** (FPR) $= \frac{FP}{FP+TN}$ ;
- **F-score** $= \frac{2\,\text{précision}\cdot\text{rappel}}{\text{précision}+\text{rappel}}$ ;
- **courbe ROC** : rappel vs fall-out quand on fait varier le seuil sur un score ; **AUC** = aire sous la courbe (0,5 = hasard, 1 = parfait).

Le taux d'erreur seul est trompeur quand les classes sont **déséquilibrées** (prédire toujours « sain » donne 99,9999 % de réussite pour une maladie rare).

**Convention du cours** : $C_{i,j}$ = nombre de points de la classe $j$ prédits dans la classe $i$ (lignes = prédiction, colonnes = vérité). Attention, scikit-learn utilise la convention transposée.

### Exercice 5.1 — Métriques

In [ ]:
def confusion(y_true, y_pred, n_classes=2):
    """C[i, j] = #{points de classe j prédits i} (convention du cours)."""
    # TODO : np.add.at(C, (y_pred, y_true), 1)
    raise NotImplementedError("À compléter")


def binary_metrics(y_true, y_pred):
    """Retourne un dict : precision, recall, fallout, f1 (classe positive = 1)."""
    C = confusion(y_true, y_pred)
    TP, FP, FN, TN = C[1, 1], C[1, 0], C[0, 1], C[0, 0]
    # TODO : définitions de l'énoncé ; renvoyer 0 quand un dénominateur est nul
    raise NotImplementedError("À compléter")


def roc_curve_np(y_true, score):
    """Retourne (fpr, tpr) pour tous les seuils, de (0, 0) à (1, 1)."""
    y_true = np.asarray(y_true); score = np.asarray(score, float)
    # TODO : trier par score décroissant, cumuler TP et FP, ne garder que les changements de seuil
    raise NotImplementedError("À compléter")


def auc(fpr, tpr):
    return float(np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2))

In [ ]:
# Tests vs scikit-learn
yt = rng.integers(0, 2, 300); yp = np.where(rng.random(300) < 0.8, yt, 1 - yt); sc = yt + rng.normal(0, 0.8, 300)
check(np.array_equal(confusion(yt, yp), confusion_matrix(yt, yp).T), "confusion = sklearn transposée")
mtr = binary_metrics(yt, yp)
check_close([mtr["precision"], mtr["recall"], mtr["f1"]],
            [precision_score(yt, yp), recall_score(yt, yp), f1_score(yt, yp)], "précision / rappel / F1 = sklearn")
check(np.isclose(auc(*roc_curve_np(yt, sc)), roc_auc_score(yt, sc)), "AUC = sklearn")
check(np.isclose(auc(*roc_curve_np(yt, np.round(sc))), roc_auc_score(yt, np.round(sc))), "AUC avec scores ex aequo")

## Partie 6 — Application : la semaine prochaine sera-t-elle agitée ?

La volatilité est **persistante** (*clustering*) : contrairement au signe des rendements, elle est largement prévisible. On pose un problème de classification **déséquilibré** :
- cible : $y_t = 1$ si la volatilité réalisée des **5 prochains jours** ($r_{t+1},\dots,r_{t+5}$) dépasse le quantile 90 % observé sur la période d'apprentissage (≈ 10 % de positifs) ;
- variables (connues en $t$) : log de la volatilité réalisée sur 5, 22 et 66 jours (semaine, mois, trimestre — l'esprit du modèle **HAR**, Corsi 2009), standardisées.

**Découpage temporel** : apprentissage 2000–2018, test 2019–2024 (jamais utilisé pour choisir quoi que ce soit).

In [ ]:
spy, src = tp_data.load_index_returns("SPY", start="2000-01-01")
print("source :", src)
feat = pd.DataFrame({"log vol 5j": np.log(spy.rolling(5).std()),
                     "log vol 22j": np.log(spy.rolling(22).std()),
                     "log vol 66j": np.log(spy.rolling(66).std())})
fwd_vol = spy[::-1].rolling(5).std()[::-1].shift(-1)          # vol de r_{t+1..t+5}
data = pd.concat([feat, fwd_vol.rename("fwd_vol")], axis=1).dropna()
train = data.loc[:"2018"]; test = data.loc["2019":]
thr = train["fwd_vol"].quantile(0.9)
mu, sd = train[feat.columns].mean(), train[feat.columns].std()
Xa, ya = ((train[feat.columns] - mu) / sd).to_numpy(), (train["fwd_vol"] > thr).astype(int).to_numpy()
Xb, yb = ((test[feat.columns] - mu) / sd).to_numpy(), (test["fwd_vol"] > thr).astype(int).to_numpy()
print(f"apprentissage : {len(ya)} jours ({ya.mean():.1%} positifs) | test : {len(yb)} jours ({yb.mean():.1%} positifs)")

### Exercice 6.1 — La fuite d'information de la validation croisée « mélangée »

Des jours consécutifs ont des variables presque identiques (les fenêtres glissantes se chevauchent) **et** des cibles qui partagent 4 jours sur 5. Avec un $r$-fold mélangé, le voisin le plus proche d'un jour de validation est souvent… la veille ou le lendemain, présent dans l'apprentissage : l'erreur est **sous-estimée**, et $k$ est mal choisi.

Remède : des blocs **contigus dans le temps**, avec un **embargo** (on retire de l'apprentissage les jours trop proches du bloc de validation). Codez `blocked_splits(n, r, embargo)` : $r$ blocs contigus ; pour chaque bloc de validation, l'apprentissage est tout le reste **sauf** `embargo` jours de part et d'autre.

On mesure le score par l'**AUC** (classes déséquilibrées), à partir de `knn_proba`.

In [ ]:
def blocked_splits(n, r, embargo):
    # TODO : blocs = np.array_split(np.arange(n), r) ; apprentissage = indices < début − embargo ou > fin + embargo
    raise NotImplementedError("À compléter")


def cv_auc(X, y, k, splits):
    scores = np.empty(len(y)); mask = np.zeros(len(y), bool)
    for tr, va in splits:
        scores[va] = knn_proba(X[tr], y[tr], X[va], k); mask[va] = True
    return roc_auc_score(y[mask], scores[mask])

ks_v = [1, 5, 15, 31, 61, 121, 241, 481, 961]
shuffled = kfold_splits(len(ya), 5, np.random.default_rng(0))
blocked = blocked_splits(len(ya), 5, embargo=22)
cv_shuf = [cv_auc(Xa, ya, k, shuffled) for k in ks_v]
cv_block = [cv_auc(Xa, ya, k, blocked) for k in ks_v]
test_auc = [roc_auc_score(yb, knn_proba(Xa, ya, Xb, k)) for k in ks_v]

fig, ax = plt.subplots()
ax.semilogx(ks_v, cv_shuf, marker="o", label="CV 5-fold mélangée")
ax.semilogx(ks_v, cv_block, marker="o", label="CV 5 blocs + embargo 22 j")
ax.semilogx(ks_v, test_auc, marker="o", label="AUC test 2019–2024 (jamais vue)", color=AQUA)
ax.set_xlabel("k"); ax.set_ylabel("AUC"); ax.legend(); ax.set_title("Validation croisée sur série temporelle")
plt.show()
k_shuf, k_block = ks_v[int(np.argmax(cv_shuf))], ks_v[int(np.argmax(cv_block))]
print(f"k choisi — CV mélangée : {k_shuf} | CV par blocs : {k_block}")
observe(cv_shuf[0] > cv_block[0] + 0.05, "la CV mélangée surestime nettement la performance du 1-NN (fuite)")
observe(max(cv_shuf) > max(test_auc), "la CV mélangée est trop optimiste par rapport au test")

### Exercice 6.2 — Évaluation finale sur 2019–2024

Comparez, sur le test, le k-NN (k choisi par CV par blocs) à une **règle simple** : score = volatilité réalisée sur 22 jours. Tracez les courbes ROC, et donnez précision / rappel / F1 au seuil « proba > 0,2 ». Comparez au classifieur trivial « jamais de forte volatilité ».

In [ ]:
p_knn = knn_proba(Xa, ya, Xb, k_block)
score_rule = Xb[:, 1]                       # log vol 22 j standardisée
fig, ax = plt.subplots(figsize=(5.5, 5.2))
for i, (name, s) in enumerate([(f"{k_block}-NN", p_knn), ("règle : vol 22 j", score_rule)]):
    fpr, tpr = roc_curve_np(yb, s)
    ax.plot(fpr, tpr, label=f"{name} (AUC {auc(fpr, tpr):.3f})", color=PALETTE[i])
ax.plot([0, 1], [0, 1], color=GRAY, ls="--", lw=1, label="hasard")
ax.set_xlabel("fall-out (FPR)"); ax.set_ylabel("rappel (TPR)"); ax.set_title("ROC sur 2019–2024"); ax.legend()
plt.show()

# TODO : binary_metrics sur (p_knn > 0.2) et sur le classifieur constant 0 ; ajouter l'accuracy à chaque dict
raise NotImplementedError("À compléter")
pd.DataFrame(rows).T

**Questions.**
1. Le classifieur « toujours 0 » a une *accuracy* élevée. Pourquoi est-il pourtant inutile ? Quelle métrique le révèle ?
2. Le k-NN fait-il mieux que la simple règle « volatilité à 22 jours » ? Qu'en conclure ?
3. Pourquoi choisir le seuil 0,2 plutôt que 0,5 ?

> ✍️ *Votre réponse :* …

## Partie 7 — Synthèse : k-NN, validation, métriques

**k-NN — avantages :** très flexible (presque aucun a priori), ne demande qu'une distance (pas de coordonnées), quelques lignes de code, naturellement multi-classe, **consistant** (l'erreur du k-NN tend vers l'erreur de Bayes si $k\to\infty$, $k/n\to0$).
**Inconvénients :** coût de **prédiction** $\Theta(nd)$ par requête (TP 1 : les structures sous-linéaires dégénèrent en grande dimension), convergence lente en grande dimension (malédiction), sensibilité à l'échelle des variables, et $k$ à choisir.

| Validation | Coût (nb d'apprentissages) | Variance de l'estimation | Quand l'utiliser |
|---|---|---|---|
| Holdout | 1 | **forte** | très gros jeux de données, modèles coûteux |
| $r$-fold ($r$ = 5 ou 10) | $r$ | faible | **défaut** |
| Leave-one-out | $n$ | faible (biais minimal) | petits jeux de données, ou quand LOO est gratuit (k-NN, régression linéaire/ridge) |
| Leave-p-out | $\binom np$ | — | quasi jamais (combinatoire) |
| Monte-Carlo | $B$ | faible si $B$ grand | quand on veut découpler taille de validation et nombre de répétitions |
| **Blocs + embargo** (hors cours) | $r$ | — | **données temporelles** (finance) : obligatoire pour éviter les fuites |

**Métriques :** *accuracy* si classes équilibrées et coûts symétriques ; **précision / rappel / F1** pour une classe rare ; **ROC / AUC** pour juger un *score* indépendamment du seuil ; la matrice de confusion pour tout voir.